# Analysis for $K_{L}\to\pi^+\pi^-$ (tag: $K_{S}\to\pi^+\pi^-$) control sample in $\Delta t$ function

In [1]:
%reload_ext autoreload
%autoreload 2

import glob
import json
import time
import os
import hashlib


import ROOT
import utils.physics_utils as PU
from utils.general_utils import get_valid_files_fast, log_analysis_run, build_cut_expression
from utils.hist_utils import (
    split_variable_group,
    fill_1d_histograms,
    fill_2d_histograms,
    draw_and_save_1d_histograms,
    draw_and_save_2d_histograms,
)

ROOT.EnableImplicitMT(90)
ROOT.gStyle.SetPalette(ROOT.kRainbow)

ROOT.gSystem.Load("/workspace/KLOE/Include/Codes/libLibRec.so")
ROOT.gSystem.Load("/workspace/KLOE/Subanalysis/CPFit/libCPFit.so")
ROOT.gInterpreter.AddIncludePath("/workspace/KLOE/src/Include/Codes/inc")
ROOT.gInterpreter.AddIncludePath("/workspace/KLOE/src/Include/FortranAnalysis/inc")
ROOT.gInterpreter.AddIncludePath("/workspace/KLOE/src/Subanalysis/CPFit/inc")

ROOT.gInterpreter.Declare('#include "/workspace/KLOE/src/Include/FortranAnalysis/inc/fort_common.h"')
ROOT.gInterpreter.Declare('#include "/workspace/KLOE/src/Include/Codes/inc/const.h"')
ROOT.gInterpreter.Declare('#include "/workspace/KLOE/src/Include/Codes/inc/HypothesisCodes.h"')
ROOT.gInterpreter.Declare('#include "/workspace/KLOE/src/Include/Codes/inc/SelectionCodes.h"')

ROOT.gInterpreter.Declare('#include "/workspace/KLOE/src/Include/Codes/inc/charged_mom.h"')
ROOT.gInterpreter.Declare('#include "/workspace/KLOE/src/Include/Codes/inc/kloe_class.h"')

ROOT.gInterpreter.Declare('#include "/workspace/KLOE/src/Subanalysis/CPFit/inc/cpfit.hpp"')

ROOT.KLOE.setGlobalStyle()

channName = {int(k): str(v) for k, v in ROOT.KLOE.channName}
channTitle = {str(k): str(v) for k, v in ROOT.KLOE.channTitle}
channColor = {str(k): int(v) for k, v in ROOT.KLOE.channColor}

# Enrichment for control samples
channTitle["hasKsPM"] = "Event includes K_{S}#rightarrowpi^{+}pi^{-}"
channTitle["noKsPM"] = "Event does not include K_{S}#rightarrowpi^{+}pi^{-}"

channColor["hasKsPM"] = ROOT.kRed
channColor["noKsPM"] = ROOT.kBlue

print(channName)

{0: 'Data', 1: 'Signal', 2: 'Regeneration', 3: 'Omega', 4: '3pi0', 5: 'Semileptonic', 6: 'Other', 7: 'pi+pi-pi+pi-', 8: 'MC sum'}


In [ ]:
# Pobierz wszystkie pliki pasujące do wzorca
all_files = glob.glob(
    "/workspace/root_files/hal/ALL_PHYS_FOUR_PI_NoSmearing/*.root"
)
all_files += glob.glob(
    "/workspace/root_files/hal/ALL_PHYS2_FOUR_PI_NoSmearing/*.root"
)
all_files += glob.glob(
    "/workspace/root_files/hal/ALL_PHYS3_FOUR_PI_NoSmearing/*.root"
)

data_files = glob.glob(
  f"/workspace/root_files/hal/DATA_FOUR_PI_NoSmearing/*.root"
)

# Użycie (błyskawiczne!):
all_files = get_valid_files_fast(all_files)
data_files = get_valid_files_fast(data_files)

# Scaling factor
lumi_data = 1634433.9750
lumi_mc = 1444990.0313
scaling_factor = lumi_data / lumi_mc
print(f"Lumi factor: {scaling_factor:.4f}")

# Przekaż czystą listę plików bezpośrednio do RDataFrame
df = ROOT.RDataFrame("h1", all_files)
df_data = ROOT.RDataFrame("h1", data_files)

mPi0 = 134.9768 # MeV
mPiCh = 139.57039 # MeV
mK0 = 497.611 # MeV

# --- PRZETWARZANIE DANYCH (DATA) ---
df_data = (
    df_data.Define("KchrecKS_5", f"KchrecKS[5] - {mK0}")
    .Define("KchrecKL_5", f"KchrecKL[5] - {mK0}")
    .Define("rho_KS", "calc_cylindrical_radius(KchrecKS, ipKS)")
    .Define("rho_KL", "calc_cylindrical_radius(KchrecKL, ipKL)")
    .Define("z_KS", "calc_z_component(KchrecKS, ipKS)")
    .Define("z_KL", "calc_z_component(KchrecKL, ipKL)")
    ######################### Lorentz transformed variables ############################
    .Define("trk1KS_phiCM", "lorentz_transform(trk1KS, Broots, Bpx, Bpy, Bpz)")
    .Define("trk2KS_phiCM", "lorentz_transform(trk2KS, Broots, Bpx, Bpy, Bpz)")
    .Define("trk1KL_phiCM", "lorentz_transform(trk1KL, Broots, Bpx, Bpy, Bpz)")
    .Define("trk2KL_phiCM", "lorentz_transform(trk2KL, Broots, Bpx, Bpy, Bpz)")
    .Define("KchrecKS_phiCM", "lorentz_transform(KchrecKS, Broots, Bpx, Bpy, Bpz)")
    .Define("KchrecKL_phiCM", "lorentz_transform(KchrecKL, Broots, Bpx, Bpy, Bpz)")
    .Define("KchboostKS_phiCM", "lorentz_transform(KchboostKS, Broots, Bpx, Bpy, Bpz)")
    .Define("KchboostKL_phiCM", "lorentz_transform(KchboostKL, Broots, Bpx, Bpy, Bpz)")
    .Define("KS_vertex", "ROOT::RVec<double>{KchrecKS[6] - ipKS[0], KchrecKS[7] - ipKS[1], KchrecKS[8] - ipKS[2]}")
    .Define("KL_vertex", "ROOT::RVec<double>{KchrecKL[6] - ipKL[0], KchrecKL[7] - ipKL[1], KchrecKL[8] - ipKL[2]}")
    ######################### Variables for Omega-pi0 geometrical cut ############################
    .Define("bhabha_IP", "ROOT::RVec<double>{Bx, By, Bz}")
    .Define("bhabha_IP_changed", "ROOT::RVec<double>{Bx, By, KchrecKL[8]}")
    .Define("rho_00_closest", "calc_cylindrical_radius(KchrecKS, bhabha_IP_changed)")
    .Define("rho_pm_closest", "calc_cylindrical_radius(KchrecKL, bhabha_IP)")
    .Define("rho_omega", "hypot(rho_00_closest, rho_pm_closest)")
    .Define("z_00_closest", "KchrecKS[8] - KchrecKL[8]")
    .Define("z_pm_closest", "KchrecKL[8] - Bz")
    ###############################################################################################
    ######################### Variables for bad tracks geometrical cut ############################
    .Define("rho_pm_00", "hypot(KchrecKL[6] - KchrecKS[6], KchrecKL[7] - KchrecKS[7])")
    .Define("z_pm_00", "KchrecKL[8] - KchrecKS[8]")
    .Define("cos_trk1_K", "cosine_between_vectors(trk1KL, KchrecKL)")
    .Define("cos_trk2_K", "cosine_between_vectors(trk2KL, KchrecKL)")
    ###############################################################################################
    ######################## Delta t ##############################################################
    .Define("Delta_t_fit", "KaonKLTimeCMFitFit - KaonKSTimeCMFitFit")
    .Define("Delta_t_rec", "KaonKLTimeCMBoostBoost - KaonKSTimeCMBoostBoost")
    .Define("Delta_t_MC", "KaonNeTimeCMMC - KaonChTimeCMMC")
    ###############################################################################################
    ######################## Resolutions ##########################################################
    .Define("Delta_t_fit_resolution", "Delta_t_fit - Delta_t_MC")
    .Define("Delta_t_rec_resolution", "Delta_t_rec - Delta_t_MC")
    .Define("trk1KL_px_resolution", "trk1KL[0] - trk1KLmc[0]")
    .Define("trk2KL_px_resolution", "trk2KL[0] - trk2KLmc[0]")
    .Define("trk1KL_py_resolution", "trk1KL[1] - trk1KLmc[1]")
    .Define("trk2KL_py_resolution", "trk2KL[1] - trk2KLmc[1]")
    .Define("trk1KL_pz_resolution", "trk1KL[2] - trk1KLmc[2]")
    .Define("trk2KL_pz_resolution", "trk2KL[2] - trk2KLmc[2]")
    .Define("trk1KS_px_resolution", "trk1KS[0] - trk1KSmc[0]")
    .Define("trk2KS_px_resolution", "trk2KS[0] - trk2KSmc[0]")
    .Define("trk1KS_py_resolution", "trk1KS[1] - trk1KSmc[1]")
    .Define("trk2KS_py_resolution", "trk2KS[1] - trk2KSmc[1]")
    .Define("trk1KS_pz_resolution", "trk1KS[2] - trk1KSmc[2]")
    .Define("trk2KS_pz_resolution", "trk2KS[2] - trk2KSmc[2]")
    .Define("trk1KS_energy_resolution", "trk1KS[3] - trk1KSmc[3]")
    .Define("trk2KS_energy_resolution", "trk2KS[3] - trk2KSmc[3]")
    .Define("trk1KL_energy_resolution", "trk1KL[3] - trk1KLmc[3]")
    .Define("trk2KL_energy_resolution", "trk2KL[3] - trk2KLmc[3]")
    .Define("KchrecKS_px_resolution", "KchrecKS[0] - KchMC[0]")
    .Define("KchrecKS_py_resolution", "KchrecKS[1] - KchMC[1]")
    .Define("KchrecKS_pz_resolution", "KchrecKS[2] - KchMC[2]")
    .Define("KchrecKS_energy_resolution", "KchrecKS[3] - KchMC[3]")
    .Define("KchrecKL_px_resolution", "KchrecKL[0] - KneMC[0]")
    .Define("KchrecKL_py_resolution", "KchrecKL[1] - KneMC[1]")
    .Define("KchrecKL_pz_resolution", "KchrecKL[2] - KneMC[2]")
    .Define("KchrecKL_energy_resolution", "KchrecKL[3] - KneMC[3]")
    .Define("KchboostKS_px_resolution", "KchboostKS[0] - KchMC[0]")
    .Define("KchboostKS_py_resolution", "KchboostKS[1] - KchMC[1]")
    .Define("KchboostKS_pz_resolution", "KchboostKS[2] - KchMC[2]")
    .Define("KchboostKS_energy_resolution", "KchboostKS[3] - KchMC[3]")
    .Define("KchboostKL_px_resolution", "KchboostKL[0] - KneMC[0]")
    .Define("KchboostKL_py_resolution", "KchboostKL[1] - KneMC[1]")
    .Define("KchboostKL_pz_resolution", "KchboostKL[2] - KneMC[2]")
    .Define("KchboostKL_energy_resolution", "KchboostKL[3] - KneMC[3]")
    .Define("ip_KS_vs_KL_x", "ipKS[0] - ipKL[0]")
    .Define("ip_KS_vs_KL_y", "ipKS[1] - ipKL[1]")
    .Define("ip_KS_vs_KL_z", "ipKS[2] - ipKL[2]")
    ###############################################################################################
    .Filter("mcflag == 0 && mctruth != -1")
)

# --- PRZETWARZANIE MONTE CARLO (MC) ---
df = (
    df.Define("KchrecKS_5", f"KchrecKS[5] - {mK0}")
    .Define("KchrecKL_5", f"KchrecKL[5] - {mK0}")
    .Define("rho_KS", "calc_cylindrical_radius(KchrecKS, ipKS)")
    .Define("rho_KL", "calc_cylindrical_radius(KchrecKL, ipKL)")
    .Define("z_KS", "calc_z_component(KchrecKS, ipKS)")
    .Define("z_KL", "calc_z_component(KchrecKL, ipKL)")
    ######################### Lorentz transformed variables ############################
    .Define("trk1KS_phiCM", "lorentz_transform(trk1KS, Broots, Bpx, Bpy, Bpz)")
    .Define("trk2KS_phiCM", "lorentz_transform(trk2KS, Broots, Bpx, Bpy, Bpz)")
    .Define("trk1KL_phiCM", "lorentz_transform(trk1KL, Broots, Bpx, Bpy, Bpz)")
    .Define("trk2KL_phiCM", "lorentz_transform(trk2KL, Broots, Bpx, Bpy, Bpz)")
    .Define("KchrecKS_phiCM", "lorentz_transform(KchrecKS, Broots, Bpx, Bpy, Bpz)")
    .Define("KchrecKL_phiCM", "lorentz_transform(KchrecKL, Broots, Bpx, Bpy, Bpz)")
    .Define("KchboostKS_phiCM", "lorentz_transform(KchboostKS, Broots, Bpx, Bpy, Bpz)")
    .Define("KchboostKL_phiCM", "lorentz_transform(KchboostKL, Broots, Bpx, Bpy, Bpz)")
    .Define("KS_vertex", "ROOT::RVec<double>{KchrecKS[6] - ipKS[0], KchrecKS[7] - ipKS[1], KchrecKS[8] - ipKS[2]}")
    .Define("KL_vertex", "ROOT::RVec<double>{KchrecKL[6] - ipKL[0], KchrecKL[7] - ipKL[1], KchrecKL[8] - ipKL[2]}")
    ######################### Variables for Omega-pi0 geometrical cut ############################
    .Define("bhabha_IP", "ROOT::RVec<double>{Bx, By, Bz}")
    .Define("bhabha_IP_changed", "ROOT::RVec<double>{Bx, By, KchrecKL[8]}")
    .Define("rho_00_closest", "calc_cylindrical_radius(KchrecKS, bhabha_IP_changed)")
    .Define("rho_pm_closest", "calc_cylindrical_radius(KchrecKL, bhabha_IP)")
    .Define("rho_omega", "hypot(rho_00_closest, rho_pm_closest)")
    .Define("z_00_closest", "KchrecKS[8] - KchrecKL[8]")
    .Define("z_pm_closest", "KchrecKL[8] - Bz")
    ###############################################################################################
    ######################### Variables for bad tracks geometrical cut ############################
    .Define("rho_pm_00", "hypot(KchrecKL[6] - KchrecKS[6], KchrecKL[7] - KchrecKS[7])")
    .Define("z_pm_00", "KchrecKL[8] - KchrecKS[8]")
    .Define("cos_trk1_K", "cosine_between_vectors(trk1KL, KchrecKL)")
    .Define("cos_trk2_K", "cosine_between_vectors(trk2KL, KchrecKL)")
    ###############################################################################################
    ######################## Delta t ##############################################################
    .Define("Delta_t_fit", "KaonKLTimeCMFitFit - KaonKSTimeCMFitFit")
    .Define("Delta_t_rec", "KaonKLTimeCMBoostBoost - KaonKSTimeCMBoostBoost")
    .Define("Delta_t_MC", "KaonNeTimeCMMC - KaonChTimeCMMC")
    ###############################################################################################
    ######################## Resolutions ##########################################################
    .Define("Delta_t_fit_resolution", "Delta_t_fit - Delta_t_MC")
    .Define("Delta_t_rec_resolution", "Delta_t_rec - Delta_t_MC")
    .Define("trk1KL_px_resolution", "trk1KL[0] - trk1KLmc[0]")
    .Define("trk2KL_px_resolution", "trk2KL[0] - trk2KLmc[0]")
    .Define("trk1KL_py_resolution", "trk1KL[1] - trk1KLmc[1]")
    .Define("trk2KL_py_resolution", "trk2KL[1] - trk2KLmc[1]")
    .Define("trk1KL_pz_resolution", "trk1KL[2] - trk1KLmc[2]")
    .Define("trk2KL_pz_resolution", "trk2KL[2] - trk2KLmc[2]")
    .Define("trk1KS_px_resolution", "trk1KS[0] - trk1KSmc[0]")
    .Define("trk2KS_px_resolution", "trk2KS[0] - trk2KSmc[0]")
    .Define("trk1KS_py_resolution", "trk1KS[1] - trk1KSmc[1]")
    .Define("trk2KS_py_resolution", "trk2KS[1] - trk2KSmc[1]")
    .Define("trk1KS_pz_resolution", "trk1KS[2] - trk1KSmc[2]")
    .Define("trk2KS_pz_resolution", "trk2KS[2] - trk2KSmc[2]")
    .Define("trk1KS_energy_resolution", "trk1KS[3] - trk1KSmc[3]")
    .Define("trk2KS_energy_resolution", "trk2KS[3] - trk2KSmc[3]")
    .Define("trk1KL_energy_resolution", "trk1KL[3] - trk1KLmc[3]")
    .Define("trk2KL_energy_resolution", "trk2KL[3] - trk2KLmc[3]")
    .Define("KchrecKS_px_resolution", "KchrecKS[0] - KchMC[0]")
    .Define("KchrecKS_py_resolution", "KchrecKS[1] - KchMC[1]")
    .Define("KchrecKS_pz_resolution", "KchrecKS[2] - KchMC[2]")
    .Define("KchrecKS_energy_resolution", "KchrecKS[3] - KchMC[3]")
    .Define("KchrecKL_px_resolution", "KchrecKL[0] - KneMC[0]")
    .Define("KchrecKL_py_resolution", "KchrecKL[1] - KneMC[1]")
    .Define("KchrecKL_pz_resolution", "KchrecKL[2] - KneMC[2]")
    .Define("KchrecKL_energy_resolution", "KchrecKL[3] - KneMC[3]")
    .Define("KchboostKS_px_resolution", "KchboostKS[0] - KchMC[0]")
    .Define("KchboostKS_py_resolution", "KchboostKS[1] - KchMC[1]")
    .Define("KchboostKS_pz_resolution", "KchboostKS[2] - KchMC[2]")
    .Define("KchboostKS_energy_resolution", "KchboostKS[3] - KchMC[3]")
    .Define("KchboostKL_px_resolution", "KchboostKL[0] - KneMC[0]")
    .Define("KchboostKL_py_resolution", "KchboostKL[1] - KneMC[1]")
    .Define("KchboostKL_pz_resolution", "KchboostKL[2] - KneMC[2]")
    .Define("KchboostKL_energy_resolution", "KchboostKL[3] - KneMC[3]")
    .Define("ip_KS_vs_KL_x", "ipKS[0] - ipKL[0]")
    .Define("ip_KS_vs_KL_y", "ipKS[1] - ipKL[1]")
    .Define("ip_KS_vs_KL_z", "ipKS[2] - ipKL[2]")
    ###############################################################################################    
    .Filter("mcflag == 1 && mctruth != -1")
)

# df Signal
df_Signal = df.Filter(
    f"mcflag == 1 && mctruth == 7"
)

df_Background = df.Filter(
    f"mcflag == 1 && mctruth != 7 && mctruth != -1"
)

columns = df_data.GetColumnNames()

print(f"Liczba dostępnych kolumn: {len(columns)}")
print("[")
for col in columns:
  print(f'"{col}",')
print("]")

Przetestowano 353 plików. Poprawnych: 351, Odrzuconych: 2
Przetestowano 420 plików. Poprawnych: 420, Odrzuconych: 0
Lumi factor: 1.1311
Liczba dostępnych kolumn: 360
[
"Asscl",
"Assleng",
"Asstr",
"Bpx",
"Bpy",
"Bpz",
"Broots",
"Bx",
"By",
"Bz",
"Chi2NeutralKinFit",
"Chi2OmegaKinFit",
"Chi2PMKLKinFit",
"Chi2PMKSKinFit",
"Chi2PMKinFit",
"Chi2SignalKinFit",
"Chi2TriKinFit",
"Cotv",
"Cotv1",
"Cotv2",
"CotvMC",
"CotvSmeared1",
"CotvSmeared2",
"Curv",
"Curv1",
"Curv2",
"CurvMC",
"CurvSmeared1",
"CurvSmeared2",
"Delta_t_MC",
"Delta_t_fit",
"Delta_t_fit_resolution",
"Delta_t_rec",
"Delta_t_rec_resolution",
"Eclfilfo",
"Eclfilfoword",
"Enecl",
"ErrorsNeutral",
"ErrorsNeutralFit",
"ErrorsOmega",
"ErrorsOmegaFit",
"ErrorsPM",
"ErrorsPMFit",
"ErrorsPMKL",
"ErrorsPMKLFit",
"ErrorsPMKS",
"ErrorsPMKSFit",
"ErrorsSignal",
"ErrorsSignalFit",
"KaonChTimeCMBoostLor",
"KaonChTimeCMBoostRec",
"KaonChTimeCMBoostTriFit",
"KaonChTimeCMMC",
"KaonChTimeCMRecLor",
"KaonChTimeCMRecRec",
"KaonChTimeCMRecTriFit",


## Optimization of the cut for the control sample tag $K_{S}\to\pi^+\pi^-$

In [16]:
analysis_name = "kl_pm_control_sample"

current_date = time.strftime("%Y-%m-%d")
current_result_dir = f"results/{analysis_name}/{current_date}"

metadata = {
    "saved_errors": "pi+pi-pi+pi-",
    "cutset": "NONE",
    "luminosity_pb_MC": lumi_mc,
    "luminosity_pb_data": lumi_data,
    "scale_factor": scaling_factor,
    "timestamp": current_date,
}

os.makedirs(current_result_dir, exist_ok=True)
with open(f"{current_result_dir}/analysis_config.json", "w") as f:
  json.dump(metadata, f, indent=4)

In [ ]:
# Generate overlapped histograms for all channels
variables_1D = ["KchrecKS_5",
                "KchrecKL_5",
                "rho_KS",
                "rho_KL",
                "z_KS",
                "z_KL",
                "bhabha_IP",
                "rho_00_closest",
                "z_00_closest",
                "rho_pm_closest",
                "z_pm_closest"
                ]
variables_1D_titles = ["m^{inv}_{+-,1} - m_{K^{0}} [MeV/c^{2}]", 
                       "m^{inv}_{+-,2} - m_{K^{0}} [MeV/c^{2}]", 
                       "#rho_{1} [cm]",
                       "#rho_{2} [cm]",
                       "z_{1} [cm]",
                       "z_{2} [cm]",
                       "m^{inv,fit,+-00}_{#gamma#gamma,1} - m_{#pi^{0}} [MeV/c^{2}]",
                       "m^{inv,fit,+-00}_{#gamma#gamma,2} - m_{#pi^{0}} [MeV/c^{2}]",
                       "m^{inv,fit,00}_{#gamma#gamma,1} - m_{#pi^{0}} [MeV/c^{2}]",
                       "m^{inv,fit,00}_{#gamma#gamma,2} - m_{#pi^{0}} [MeV/c^{2}]",
                       "m^{inv,fit,+-}_{#gamma#gamma,1} - m_{#pi^{0}} [MeV/c^{2}]",
                       "m^{inv,fit,+-}_{#gamma#gamma,2} - m_{#pi^{0}} [MeV/c^{2}]",
                       "m^{inv,rec}_{#gamma#gamma,1} - m_{#pi^{0}} [MeV/c^{2}]",
                       "m^{inv,rec}_{#gamma#gamma,2} - m_{#pi^{0}} [MeV/c^{2}]",
                       "#Deltat^{fit} [#tau_{S}]",
                       "#Deltat^{tri,fit} [#tau_{S}]",
                       "#Deltat^{rec} [#tau_{S}]",
                       "Total number of clusters",
                       "#Delta_{6} [cm^{2}]",
                       "Number of neutral clusters"
                       ]
range_X_1D = [(5000, -400, 200), 
              (100, -400, 200), 
              (100, -200, 200),
              (100, 0, 500),
              (5000, 0, 500),
              (100, 0, 100),
              (100, -400, 400),
              (100, -400, 400),
              (100, -400, 400),
              (100, -400, 400),
              (100, -400, 400),
              (100, -400, 400),
              (100, -400, 400),
              (100, -400, 400),
              (401, -300, 300),
              (401, -300, 300),
              (401, -300, 300),
              (100, 0, 20),
              (5000, 0, 10000),
              (100, 0, 20)
              ]

# Zmienne widoczne TYLKO dla jednego, wybranego mctruth (np. prawdziwe wartości z MC,
# dostępne tylko w próbce sygnałowej) - bez Data i bez MC sum. Można wskazać
# dowolne mctruth (niekoniecznie 1) ustawiając TRUTH_MCTRUTH_ID_1D.
TRUTH_ONLY_VARS_1D = {}
TRUTH_MCTRUTH_ID_1D = 1